In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error,mean_squared_error
import xgboost as xgb

In [2]:
#importing dataset
df = pd.read_csv("car_fuel_efficiency_2026.csv")

In [3]:
#filling missing values with "0"
df = df.fillna(0)
#Splitting the dataset
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [4]:
#Splitting train and target dataset
X_train_origin = df_train.drop(columns = "fuel_efficiency_mpg")
y_train = df_train.fuel_efficiency_mpg.values

X_val_origin = df_val.drop(columns = "fuel_efficiency_mpg")
y_val = df_val.fuel_efficiency_mpg.values

In [5]:
X_val_origin.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration'],
      dtype='str')

In [6]:
#One-hot encoding categorical features
dv = DictVectorizer(sparse= True)
X_train = dv.fit_transform(X_train_origin.to_dict(orient="records"))
X_val = dv.fit_transform(X_val_origin.to_dict(orient="records"))


Decision Tree Regressor

In [7]:
#building DecisionTreeRegressor
dtr = DecisionTreeRegressor(max_depth=1)
dtr.fit(X_train,y_train)
print(export_text(dtr,feature_names=dv.get_feature_names_out()))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [8]:
#Therefore, the original feature that is used for splitting the data is "model_year"

Random Forest Regressor

In [9]:
#Building RandomForestRegressor model
rfr = RandomForestRegressor(n_estimators=10,random_state=1,n_jobs=-1)
rfr.fit(X_train,y_train)
predictions = rfr.predict(X_val)

#Calculating the model's RMSE
rfr_rmse = root_mean_squared_error(y_val,predictions)
print("the RMSE of RandomForestRegressor model on the validation data: %.4f" % rfr_rmse)


the RMSE of RandomForestRegressor model on the validation data: 1.8332


In [10]:
#experiment with the `n_estimators` parameter
result = {}
estimators = [10, 50, 100, 150]
for estimator in estimators:
    rfr = RandomForestRegressor(n_estimators=estimator,random_state=1,n_jobs=-1)
    rfr.fit(X_train,y_train)
    predictions = rfr.predict(X_val)

    #Calculating the model's RMSE
    rfr_rmse = root_mean_squared_error(y_val,predictions)
    result[estimator] = round(rfr_rmse,3)
result

{10: 1.833, 50: 1.774, 100: 1.77, 150: 1.769}

In [11]:
#the best n_estimator 
best_estimator = max(result.items(), key= lambda item:item[1])
print(f"The best 'n_estimator' parameter is {best_estimator}.")

The best 'n_estimator' parameter is (10, 1.833).


In [12]:
#experiment with the `max_depth` parameter
scores = []
depths = [10, 15, 20, 25]
estimators = [10, 50, 100, 150]
for depth in depths:
    for estimator in estimators:
        rfr = RandomForestRegressor(max_depth=depth,n_estimators=estimator,random_state=1,n_jobs=-1)
        rfr.fit(X_train,y_train)
        predictions = rfr.predict(X_val)

        #Calculating the model's RMSE
        rfr_rmse = root_mean_squared_error(y_val,predictions)
        scores.append((depth,estimator,round(rfr_rmse,4)))

In [13]:
scores = pd.DataFrame(scores,columns = ["max-depth","n-estimator","RMSE"])
scores.sort_values(by="RMSE",ascending = False)

,max-depth,n-estimator,RMSE
8,20,10,1.8373
12,25,10,1.8369
4,15,10,1.8314
0,10,10,1.7823
9,20,50,1.7724
13,25,50,1.7721
5,15,50,1.7709
10,20,100,1.7689
11,20,150,1.7685
14,25,100,1.7683


Feature Importance

In [14]:
X_train_subset = X_train_origin[['vehicle_weight',
'horsepower',
'acceleration',
'engine_displacement']]
dv_subset = DictVectorizer(sparse=True)
subset_train = dv_subset.fit_transform(X_train_subset.to_dict(orient="records"))
features_subset = dv_subset.get_feature_names_out()

In [15]:
modified_rfr = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
modified_rfr.fit(subset_train,y_train)
feature_importance = modified_rfr.feature_importances_

In [16]:
list(zip(features_subset,feature_importance))

[('acceleration', np.float64(0.20394456683868664)),
 ('engine_displacement', np.float64(0.21444459482965664)),
 ('horsepower', np.float64(0.2295164451133977)),
 ('vehicle_weight', np.float64(0.35209439321825897))]

XGBoost model

In [21]:
features = list(dv.get_feature_names_out())
features

['acceleration',
 'drivetrain=All-wheel drive',
 'drivetrain=Front-wheel drive',
 'drivetrain=Rear-wheel drive',
 'engine_displacement',
 'fuel_type=Diesel',
 'fuel_type=Gasoline',
 'fuel_type=Hybrid',
 'horsepower',
 'model_year',
 'num_cylinders',
 'num_doors',
 'origin=Asia',
 'origin=Europe',
 'origin=USA',
 'vehicle_weight']

In [22]:
dtrain = xgb.DMatrix(X_train,label=y_train,feature_names=features)
dval = xgb.DMatrix(X_val,label=y_val,feature_names=features)

In [23]:
watchlist = [(dtrain,"train"),(dval,"val")]

In [24]:
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
xgb_model = xgb.train(xgb_params, dtrain,evals=watchlist,num_boost_round=100)

[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312
[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512
[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[24]	train-rmse:1.36071	va

In [35]:
y_pred = xgb_model.predict(dval)
rmse_xgb_03 = root_mean_squared_error(y_val,y_pred)
print("eta-0.3 %.4f"% rmse_xgb_03)

eta-0.3 1.8265


In [30]:
xgb_params_2 = {
    'eta': 0.1, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
xgb_model_2 = xgb.train(xgb_params_2, dtrain,evals=watchlist,num_boost_round=100)

[0]	train-rmse:2.74739	val-rmse:2.78635
[1]	train-rmse:2.58527	val-rmse:2.63162
[2]	train-rmse:2.44540	val-rmse:2.50036
[3]	train-rmse:2.32187	val-rmse:2.38568
[4]	train-rmse:2.21445	val-rmse:2.28765
[5]	train-rmse:2.12109	val-rmse:2.20428
[6]	train-rmse:2.03996	val-rmse:2.13241
[7]	train-rmse:1.96795	val-rmse:2.06798
[8]	train-rmse:1.90640	val-rmse:2.01618
[9]	train-rmse:1.85240	val-rmse:1.96998
[10]	train-rmse:1.80592	val-rmse:1.93152
[11]	train-rmse:1.76410	val-rmse:1.89688
[12]	train-rmse:1.72781	val-rmse:1.86726
[13]	train-rmse:1.69564	val-rmse:1.84312
[14]	train-rmse:1.66745	val-rmse:1.82218
[15]	train-rmse:1.64286	val-rmse:1.80346
[16]	train-rmse:1.62187	val-rmse:1.79038
[17]	train-rmse:1.60321	val-rmse:1.77726
[18]	train-rmse:1.58694	val-rmse:1.76827
[19]	train-rmse:1.57240	val-rmse:1.75951
[20]	train-rmse:1.55847	val-rmse:1.75076
[21]	train-rmse:1.54640	val-rmse:1.74347
[22]	train-rmse:1.53643	val-rmse:1.73912
[23]	train-rmse:1.52675	val-rmse:1.73485
[24]	train-rmse:1.51763	va

In [34]:
y_pred_2 = xgb_model_2.predict(dval)
rmse_xgb_01 = root_mean_squared_error(y_val,y_pred_2)
print("eta-0.1 %.4f"% rmse_xgb_01)

eta-0.1 1.7248


In [36]:
rmse_xgb_03 > rmse_xgb_01

True